In [2]:
import pandas as pd
import numpy as np

# Load the CDC data - check all sheets first
excel_file = pd.ExcelFile('./data/raw/cdc_251644_DS1.xlsx')
print("Available sheets:")
for i, sheet in enumerate(excel_file.sheet_names):
    print(f"{i}: {sheet}")

Available sheets:
0: READ ME
1: Profile of Acute Care Hospitals
2: Table of Contents
3: Table 1a-CLABSI
4: Table 1b-CAUTI
5: Table 1c-VAE 
6: Table 1d-COLO
7: Table 1d-HYST
8: Table 1e-MRSA
9: Table 1f-CDI
10: Table 1g Footnotes
11: Table 2a-NAT'L DA Data 
12: Table 2a-i-NAT'L DA Data 
13: Table 2a-ii-NAT'L DA Data
14: Table 2a-iii-NAT'L DA Data  
15: Table 2b-NAT'L LABID Data
16: Table 2c-NAT'L SSI Data
17: Table 2d-NAT'L SSI Data
18: Table 3a-State CLABSI Data
19: Table 3b-State CLABSI Data
20: Table 3c-State CLABSI Data
21: Table 3d-State CLABSI Data
22: Table 4a-State CAUTI Data
23: Table 4b-State CAUTI Data
24: Table 4c-State CAUTI Data
25: Table 5a-State VAE Data 
26: Table 5b-State VAE Data 
27: Table 5c-State VAE Data 
28: Table 6a-State SSI Data
29: Table 6b-State SSI Data
30: Table 6c-State SSI Data
31: Table 6d-State SSI Data
32: Table 6e-State SSI Data
33: Table 6f-State SSI Data
34: Table 6g-State SSI Data
35: Table 6h-State SSI Data
36: Table 6i-State SSI Data
37: Table 6

In [3]:
# Load state-level SIR comparison data 
# SIR > 1 means worse than national average, < 1 means better

# Read multiple state SIR sheets and combine
sir_data = []

for sheet_num in range(46, 53):  # Tables 10a through 10g
    sheet_name = excel_file.sheet_names[sheet_num]
    print(f"\nLoading {sheet_name}...")
    
    df = pd.read_excel('./data/raw/cdc_251644_DS1.xlsx', sheet_name=sheet_name, header=1)
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()[:5]}")  # First 5 columns
    print(df.head(3))


Loading Table 10a-State SIR Comparison...
Shape: (65, 6)
Columns: ['10a. Central line-associated bloodstream infections (CLABSI), all locations1', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
  10a. Central line-associated bloodstream infections (CLABSI), all locations1  \
0                                                NaN                             
1                                             State2                             
2                                            Alabama                             

                                     Unnamed: 1 Unnamed: 2       Unnamed: 3  \
0    All Acute Care Hospitals Reporting to NHSN        NaN              NaN   
1                                      2022 SIR   2023 SIR  Percent Change3   
2                                         1.035      0.919             0.11   

                                          Unnamed: 4 Unnamed: 5  
0                                                NaN        NaN  
1  Direction of Ch

In [5]:
# Column 0 = State names
# Column 1 = 2022 SIR 
# Column 2 = 2023 SIR 
# Column 3 = % Change
# Column 4 = Direction
# Column 5 = p value

# Rename columns to something readable
df_clabsi.columns = ['State', 'SIR_2022', 'SIR_2023', 'Percent_Change', 'Direction', 'P_Value']

print("New column names:")
print(df_clabsi.columns.tolist())
print("\nFirst 5 rows:")
print(df_clabsi.head())

New column names:
['State', 'SIR_2022', 'SIR_2023', 'Percent_Change', 'Direction', 'P_Value']

First 5 rows:
      State  SIR_2022  SIR_2023   Percent_Change  \
0    State2  2022 SIR  2023 SIR  Percent Change3   
1   Alabama     1.035     0.919             0.11   
2    Alaska     0.375     0.512             0.37   
3   Arizona     0.759     0.677             0.11   
4  Arkansas     0.816     0.667             0.18   

                                           Direction  P_Value  
0  Direction of Change, Based on Statistical Sign...  p-value  
1                                          No change   0.0689  
2                                          No change   0.3532  
3                                          No change   0.0973  
4                                           Decrease   0.0344  


In [6]:
# fix row 0
df_clabsi = df_clabsi.iloc[1:]

print("After removing header row:")
print(df_clabsi.head())

After removing header row:
        State SIR_2022 SIR_2023 Percent_Change  Direction P_Value
1     Alabama    1.035    0.919           0.11  No change  0.0689
2      Alaska    0.375    0.512           0.37  No change  0.3532
3     Arizona    0.759    0.677           0.11  No change  0.0973
4    Arkansas    0.816    0.667           0.18   Decrease  0.0344
5  California    0.835    0.751            0.1   Decrease  0.0004


In [7]:
# convert SIR_2023 values from txt to number 
df_clabsi['SIR_2023'] = pd.to_numeric(df_clabsi['SIR_2023'], errors='coerce')

print("Data types:")
print(df_clabsi.dtypes)
print("\nFirst 5 rows:")
print(df_clabsi.head())
print("\nAny missing values?")
print(df_clabsi.isnull().sum())

Data types:
State                 str
SIR_2022           object
SIR_2023          float64
Percent_Change     object
Direction             str
P_Value            object
dtype: object

First 5 rows:
        State SIR_2022  SIR_2023 Percent_Change  Direction P_Value
1     Alabama    1.035     0.919           0.11  No change  0.0689
2      Alaska    0.375     0.512           0.37  No change  0.3532
3     Arizona    0.759     0.677           0.11  No change  0.0973
4    Arkansas    0.816     0.667           0.18   Decrease  0.0344
5  California    0.835     0.751            0.1   Decrease  0.0004

Any missing values?
State              2
SIR_2022           8
SIR_2023          10
Percent_Change     8
Direction          8
P_Value            8
dtype: int64


In [8]:
# I only need State and SIR_2023 for the dashboard
df_clabsi_clean = df_clabsi[['State', 'SIR_2023']].copy()

# Remove rows where SIR_2023 is missing
df_clabsi_clean = df_clabsi_clean.dropna(subset=['SIR_2023'])

# Rename the SIR_2023 column to include the infection name
df_clabsi_clean.columns = ['State', 'CLABSI_SIR_2023']

print("Clean CLABSI data:")
print(df_clabsi_clean.head(10))
print(f"\nShape: {df_clabsi_clean.shape}")

Clean CLABSI data:
          State  CLABSI_SIR_2023
1       Alabama            0.919
2        Alaska            0.512
3       Arizona            0.677
4      Arkansas            0.667
5    California            0.751
6      Colorado            0.565
7   Connecticut            0.613
8          D.C.            0.729
9      Delaware            0.765
10      Florida            0.590

Shape: (53, 2)


In [9]:
# Define a function that cleans infection data

def extract_infection_data(sheet_name, infection_name):
    """
    Load and clean one infection type from CDC data.
    Returns: DataFrame with State and SIR_2023 columns
    """
    
    # Read the Excel sheet (skip first 2 rows of headers)
    df = pd.read_excel(
        './data/raw/cdc_251644_DS1.xlsx',
        sheet_name=sheet_name,
        skiprows=2
    )
    
    # Rename columns
    df.columns = ['State', 'SIR_2022', 'SIR_2023', 'Percent_Change', 'Direction', 'P_Value']
    
    # Remove header row (row 0)
    df = df.iloc[1:]
    
    # Convert SIR_2023 to numbers
    df['SIR_2023'] = pd.to_numeric(df['SIR_2023'], errors='coerce')
    
    # Keep only State and SIR_2023
    df_clean = df[['State', 'SIR_2023']].copy()
    
    # Remove missing values
    df_clean = df_clean.dropna(subset=['SIR_2023'])
    
    # Rename SIR_2023 to include infection name
    df_clean.columns = ['State', f'{infection_name}_SIR_2023']
    
    return df_clean

# Test the function on CAUTI
df_cauti = extract_infection_data('Table 10b-State SIR Comparison', 'CAUTI')
print("CAUTI data:")
print(df_cauti.head())

CAUTI data:
        State  CAUTI_SIR_2023
1     Alabama           0.629
2      Alaska           0.909
3     Arizona           0.405
4    Arkansas           0.440
5  California           0.715


In [10]:
infections = [
    ('Table 10a-State SIR Comparison', 'CLABSI'),
    ('Table 10b-State SIR Comparison', 'CAUTI'),
    ('Table 10c-State SIR Comparison ', 'VAE'),
    ('Table 10d-State SIR Comparison', 'SSI_COLON'),
    ('Table 10e-State SIR Comparison', 'SSI_HYST'),
    ('Table 10f-State SIR Comparison', 'MRSA'),
    ('Table 10g-State SIR Comparison', 'CDI'),
]
df_all = None

for sheet_name, infection_name in infections:
    print(f"Loading {infection_name}...")
    
    df_infection = extract_infection_data(sheet_name, infection_name)
    
    # Merge with the growing master dataframe
    if df_all is None:
        df_all = df_infection
    else:
        # Merge on State column 
        df_all = df_all.merge(df_infection, on='State', how='outer')
    
    print(f"  {infection_name}: {len(df_infection)} states")

print(f"\nFinal dataset shape: {df_all.shape}")
print(f"States: {len(df_all)}")
print(f"Columns: {df_all.columns.tolist()}")
print("\nFirst 10 rows:")
print(df_all.head(10))

Loading CLABSI...
  CLABSI: 53 states
Loading CAUTI...
  CAUTI: 53 states
Loading VAE...
  VAE: 48 states
Loading SSI_COLON...
  SSI_COLON: 52 states
Loading SSI_HYST...
  SSI_HYST: 53 states
Loading MRSA...
  MRSA: 53 states
Loading CDI...
  CDI: 53 states

Final dataset shape: (54, 8)
States: 54
Columns: ['State', 'CLABSI_SIR_2023', 'CAUTI_SIR_2023', 'VAE_SIR_2023', 'SSI_COLON_SIR_2023', 'SSI_HYST_SIR_2023', 'MRSA_SIR_2023', 'CDI_SIR_2023']

First 10 rows:
         State  CLABSI_SIR_2023  CAUTI_SIR_2023  VAE_SIR_2023  \
0      Alabama            0.919           0.629         1.025   
1       Alaska            0.512           0.909         1.814   
2       All US            0.724           0.621         1.131   
3      Arizona            0.677           0.405         0.797   
4     Arkansas            0.667           0.440         1.947   
5   California            0.751           0.715         1.182   
6     Colorado            0.565           0.527         1.215   
7  Connecticut   